In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.append('..')
from utils import MnistDataloader
from os.path  import join
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

#### Saving images as csv file

با وجود موجود بودن دیتا فرآیند تبدیل هم انجام شد جهت یادگیری

In [ ]:
# source: https://www.kaggle.com/code/hojjatk/read-mnist-dataset/notebook
# Verify Reading Dataset via MnistDataloader class
input_path = '../datasets/mnist_ubyte/'
training_images_filepath = join(input_path, 'train-images-idx3-ubyte/train-images-idx3-ubyte')
training_labels_filepath = join(input_path, 'train-labels-idx1-ubyte/train-labels-idx1-ubyte')
test_images_filepath = join(input_path, 't10k-images-idx3-ubyte/t10k-images-idx3-ubyte')
test_labels_filepath = join(input_path, 't10k-labels-idx1-ubyte/t10k-labels-idx1-ubyte')


# Helper function to show a list of images with their relating titles
def show_images(images, title_texts):
    cols = 5
    rows = int(len(images)/cols) + 1
    plt.figure(figsize=(30,20))
    index = 1    
    for x in zip(images, title_texts):        
        image = x[0]        
        title_text = x[1]
        plt.subplot(rows, cols, index)        
        plt.imshow(image, cmap=plt.cm.gray)
        if (title_text != ''):
            plt.title(title_text, fontsize = 15);        
        index += 1

# Load MINST dataset
mnist_dataloader = MnistDataloader(training_images_filepath, training_labels_filepath, test_images_filepath, test_labels_filepath)
(x_train, y_train), (x_test, y_test) = mnist_dataloader.load_data()


# Show some random training and test images 
images_2_show = []
titles_2_show = []
for i in range(0, 10):
    r = np.random.randint(1, 60000)
    images_2_show.append(x_train[r])
    titles_2_show.append('training image [' + str(r) + '] = ' + str(y_train[r]))    

for i in range(0, 5):
    r = np.random.randint(1, 10000)
    images_2_show.append(x_test[r])        
    titles_2_show.append('test image [' + str(r) + '] = ' + str(y_test[r]))    

show_images(images_2_show, titles_2_show)

In [ ]:
def convert_mnist_to_csv(x_train, y_train, x_test, y_test, train_path, test_path):
    """
    Convert MNIST data (loaded by your class) to CSV format.
    
    Args:
        x_train: List of 28x28 numpy arrays (training images)
        y_train: List/array of labels (training)
        x_test: List of 28x28 numpy arrays (test images)
        y_test: List/array of labels (test)
        train_path: Output path for training CSV
        test_path: Output path for test CSV
    """
    
    # Convert list of images to numpy array
    # x_train is a list of 28x28 arrays, we need to flatten each one
    x_train_flat = []
    for img in x_train:
        x_train_flat.append(np.array(img).flatten())
    x_train_flat = np.array(x_train_flat)
    
    x_test_flat = []
    for img in x_test:
        x_test_flat.append(np.array(img).flatten())
    x_test_flat = np.array(x_test_flat)
    
    # Convert labels to numpy array if they're not already
    y_train = np.array(y_train)
    y_test = np.array(y_test)
    
    print(f"Flattened training shape: {x_train_flat.shape}")
    print(f"Flattened test shape: {x_test_flat.shape}")
    
    # Create column names: pixel0, pixel1, ..., pixel783 (# 28 × 28 = 784 pixels total)
    pixel_columns = [f'pixel{i}' for i in range(784)]
    
    # Create DataFrames
    df_train = pd.DataFrame(x_train_flat, columns=pixel_columns)
    df_train.insert(0, 'label', y_train)
    
    df_test = pd.DataFrame(x_test_flat, columns=pixel_columns)
    df_test.insert(0, 'label', y_test)
    
    # Save to CSV
    df_train.to_csv(train_path, index=False)
    df_test.to_csv(test_path, index=False)
    
    print(f"Training CSV saved: {train_path}")
    print(f"   -{df_train.shape[0]} rows, {df_train.shape[1]} columns")
    print(f"Test CSV saved: {test_path}")
    print(f"   - {df_test.shape[0]} rows, {df_test.shape[1]} columns")
    
    return df_train, df_test

In [ ]:
mnist_dataloader = MnistDataloader(training_images_filepath, training_labels_filepath, 
                                   test_images_filepath, test_labels_filepath)
(x_train, y_train), (x_test, y_test) = mnist_dataloader.load_data()
print(f"Training data shape: {len(x_train)}")  # (60000, 28, 28)
print(f"Training labels shape: {len(y_train)}")  # (60000,)
print(f"Test data shape: {len(x_test)}")  # (10000, 28, 28)
print(f"Test labels shape: {len(y_test)}")  # (10000,)

In [ ]:
# Convert and save
df_train, df_test = convert_mnist_to_csv(x_train, y_train, x_test, y_test, train_path="train.csv", test_path="test.csv")

In [ ]:
df = pd.read_csv('../datasets/mnist_csv/mnist_test.csv') # dataset on web

In [ ]:
df.columns

In [ ]:
df_test.columns

#### KNN (MINST)

<b>KNN</b> classifies a new data point by looking at the 'K' closest points in the training data and taking a majority vote.

Imagine you're new to a city and want to find a good restaurant:

1. You ask 10 friends for recommendations (K=10)
2. Each friend tells you their favorite restaurant
3. You count which restaurant gets the most votes
4. You go to the most popular one!

<b>KNN works exactly the same way!</b>

<!-- In a Jupyter markdown cell, paste this: -->

<h3>Three Main Things to Know:</h3>

<ul>
  <li><b>K</b> = How many neighbors to look at (odd number to avoid ties)</li>
  <li><b>Distance</b> = How to measure "closeness" (usually Euclidean)</li>
  <li><b>Voting</b> = Majority vote (or weighted by distance)</li>
</ul>

<h3>When to Use KNN:</h3>

<ul>
  <li>✅ Small to medium datasets</li>
  <li>✅ Clear clusters/separation</li>
  <li>✅ Low dimensions (few features)</li>
  <li>✅ Need interpretability</li>
</ul>

<p><b>KNN is simple: Your neighbors determine your class!</b></p>

# Data

In [ ]:
# seperating label and data
X_train = df_train.iloc[:, 1:]
X_test = df_test.iloc[:, 1:]
y_train = df_train.iloc[:, 0]
y_test = df_test.iloc[:, 0]

# Visualization

In [ ]:
# first digit
img = np.array(X_train.iloc[0, :]).reshape(28, 28)
plt.imshow(img)

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(10, 6))
fig.suptitle("Digit Plot")

ax_indice = np.indices((2, 3)).reshape(2, -1).T
for i, img in enumerate(ax_indice):
    row, col = ax_indice[i][0], ax_indice[i][1]
    ax = axes[row, col]
    img = X_train.iloc[i].values.reshape(28, 28)
    ax.imshow(img, cmap="gray")
    ax.set_title(f"Real digit: {y_train.iloc[i]}", fontsize=9)

# Model

| Value | Meaning |
|:------|:--------|
| n_jobs=1 | Use 1 CPU core (no parallelization) |
| n_jobs=2 | Use 2 CPU cores |
| n_jobs=-1 | Use ALL available CPU cores |
| n_jobs=-2 | Use all but 1 CPU core |
| n_jobs=-3 | Use all but 2 CPU cores |

In [ ]:
#---------------
# training model
#---------------
k = 3
knn = KNeighborsClassifier(n_neighbors=k, n_jobs=-1)
knn.fit(X_train, y_train)

#---------------
# prediction
#---------------
y_pred = knn.predict(X_test)

#---------------
# evaluation
#---------------

# accuracy score
accuracy = accuracy_score(y_test, y_pred)

# classification report
cr = classification_report(y_test, y_pred)

# confusion matrix
cm = confusion_matrix(y_test, y_pred)
total = np.sum(cm)
correct = np.trace(cm) # Return the sum along diagonals of the array
accuracy_cm = correct / total
accuracy_per_class = np.diag(cm) / np.sum(cm, axis=1)

#---------------
# result
#---------------
print(f"Accuracy Score (K={k}): {accuracy}")
print('='*50)
print(cr)
print('='*50)
print(cm)
print(f'Confusion matrix accuracy: {correct / total}')
# (TP + TN) / (TP + TN + FP + FN) -->  (sum along diagonals of the array) / (sum total)
for i, acc in enumerate(accuracy_per_class):
    print(f'Class {i}: {acc:.3f}%')


<!-- In a Jupyter markdown cell, paste this: -->

**Rows** = Actual classes (0-9)  
**Columns** = Predicted classes (0-9)  
**Diagonal** = Correct predictions (bold numbers)

| Actual \ Predicted | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 |
|:-------------------|--:|--:|--:|--:|--:|--:|--:|--:|--:|--:|
| **0** | **974** | 1 | 1 | 0 | 0 | 1 | 2 | 1 | 0 | 0 |
| **1** | 0 | **1133** | 2 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| **2** | 10 | 9 | **996** | 2 | 0 | 0 | 0 | 13 | 2 | 0 |
| **3** | 0 | 2 | 4 | **976** | 1 | 13 | 1 | 7 | 3 | 3 |
| **4** | 1 | 6 | 0 | 0 | **950** | 0 | 4 | 2 | 0 | 19 |
| **5** | 6 | 1 | 0 | 11 | 2 | **859** | 5 | 1 | 3 | 4 |
| **6** | 5 | 3 | 0 | 0 | 3 | 3 | **944** | 0 | 0 | 0 |
| **7** | 0 | 21 | 5 | 0 | 1 | 0 | 0 | **991** | 0 | 10 |
| **8** | 8 | 2 | 4 | 16 | 8 | 11 | 3 | 4 | **914** | 4 |
| **9** | 4 | 5 | 2 | 8 | 9 | 2 | 1 | 8 | 2 | **968** |

In [ ]:
k_range = np.arange(1, 22, 2)

accuracy_scores = dict()

for k in k_range:
    # train model
    knn = KNeighborsClassifier(n_neighbors=k, n_jobs=-1)
    knn.fit(X_train, y_train)
    
    # prediction
    y_pred = knn.predict(X_test)
    
    # evaluation
    accuracy = accuracy_score(y_test, y_pred)
    
    accuracy_scores[k] = accuracy

accuracy_series = pd.Series(accuracy_scores, name='accuracy')
best_k = accuracy_series.idxmax() # best k with maximum value

print(accuracy_series)
print('='*50)
print(f'The best K: {best_k}')
print(f'The best accuracy: {accuracy_series[best_k]}')

In [ ]:
plt.plot(accuracy_series.index, accuracy_series.values, 
         linestyle='--', 
         marker='o', 
         color='blue',
         markersize=8,
         linewidth=2, 
         label='Accuracy')
plt.xticks(accuracy_series.index)
plt.title('K vs Accuracy')
plt.plot(best_k, accuracy_series[best_k], marker='o', color='red', markersize=8, label='Best k')
plt.text(best_k + 0.5, accuracy_series[best_k], 'Best k')
# for i, (k, acc) in enumerate(accuracy_series.items()):
#     plt.text(k, acc, f'{acc:.2f}', ha='center', va='bottom', fontsize=10)
plt.legend()
plt.show()

#### Chat bot codes for clarifiacation:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from matplotlib.patches import Circle

# ============================================
# VISUALIZE KNN STEP-BY-STEP
# ============================================

# Create simple dataset
np.random.seed(42)
X1 = np.random.randn(30, 2) + np.array([-2, -2])
X2 = np.random.randn(30, 2) + np.array([2, 2])
X = np.vstack([X1, X2])
y = np.hstack([np.zeros(30), np.ones(30)])

# New point to classify
new_point = np.array([0.5, 0.5])

# Calculate distances to all points
distances = np.sqrt(np.sum((X - new_point)**2, axis=1))
sorted_idx = np.argsort(distances)

# Different K values to try
K_values = [1, 3, 5, 11]

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for idx, K in enumerate(K_values):
    ax = axes[idx // 2, idx % 2]
    
    # Plot training data
    ax.scatter(X[y==0][:, 0], X[y==0][:, 1], c='blue', alpha=0.6, label='Class 0')
    ax.scatter(X[y==1][:, 0], X[y==1][:, 1], c='red', alpha=0.6, label='Class 1')
    
    # Plot new point
    ax.scatter(new_point[0], new_point[1], color='green', s=200, marker='*', 
              edgecolors='black', linewidths=2, label='New Point')
    
    # Find K nearest neighbors
    k_nearest_idx = sorted_idx[:K]
    k_nearest = X[k_nearest_idx]
    k_labels = y[k_nearest_idx]
    
    # Plot K nearest neighbors with highlight
    ax.scatter(k_nearest[:, 0], k_nearest[:, 1], color='orange', s=100, 
              edgecolors='black', linewidths=2, label=f'{K} Nearest')
    
    # Draw circle showing radius to Kth neighbor
    radius = distances[sorted_idx[K-1]]
    circle = Circle((new_point[0], new_point[1]), radius, fill=False, 
                   edgecolor='black', linestyle='--', linewidth=1.5)
    ax.add_patch(circle)
    
    # Calculate prediction
    pred_class = np.argmax(np.bincount(k_labels.astype(int)))
    class_counts = np.bincount(k_labels.astype(int))
    
    # Show vote
    vote_text = f"Class 0: {class_counts[0] if len(class_counts) > 0 else 0}, "
    vote_text += f"Class 1: {class_counts[1] if len(class_counts) > 1 else 0}"
    
    ax.set_title(f'K = {K}\nPrediction: Class {int(pred_class)} | {vote_text}')
    ax.set_xlim(-5, 5)
    ax.set_ylim(-5, 5)
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.datasets import make_classification

# Create dataset with decision boundary
X, y = make_classification(n_samples=200, n_features=2, n_redundant=0,
                           n_clusters_per_class=1, random_state=42)

# Plot decision boundaries for different K
K_values = [1, 5, 15, 25]
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for idx, K in enumerate(K_values):
    ax = axes[idx // 2, idx % 2]
    
    # Train KNN
    knn = KNeighborsClassifier(n_neighbors=K)
    knn.fit(X, y)
    
    # Create mesh for decision boundary
    h = 0.02
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    # Predict on mesh
    Z = knn.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    # Plot decision boundary
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    ax.scatter(X[y==0][:, 0], X[y==0][:, 1], c='blue', alpha=0.6, label='Class 0')
    ax.scatter(X[y==1][:, 0], X[y==1][:, 1], c='red', alpha=0.6, label='Class 1')
    ax.set_title(f'K = {K}')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()